# 🔴 LLaMA 3.1 8B Overnight Fine-Tuning: Refusal-Free & Technically Accurate

**Target:** Google Colab Free Tier (**T4 GPU**)
**Runtime:** ~1.5 to 2.5 hours (safe for overnight run without disconnection)
**Dataset Composition:** ~7,400 curated high-density samples
1. `cognitivecomputations/dolphin` (2,000) — Refusal-free compliance
2. `darkknight25/Exploit_Database_Dataset` (1,500) — Real CVEs, RCEs, and exploit mechanics
3. `Vyber07/cyber-security` (1,500) — Web portal auditing, IDOR, penetration testing
4. `truthfulqa/truthful_qa` (817) — Ground-truth factuality anchoring
5. `teknium/OpenHermes-2.5` (1,500) — Multi-step technical reasoning
6. `jailfact-bench.xlsx` (99) — Local reference benchmark

---

## Cell 1: Install Dependencies (Unsloth + HuggingFace)

In [ ]:
%%capture
!pip install unsloth
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install datasets pandas openpyxl requests

print("✅ Dependencies successfully installed!")

## Cell 2: Check Hardware (T4 GPU Verification)

In [ ]:
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f"GPU: {gpu.name}")
    print(f"VRAM: {gpu.total_memory / (1024**3):.2f} GB")
else:
    raise RuntimeError("❌ No GPU detected! Please go to: Runtime > Change runtime type > T4 GPU")

## Cell 3: Load LLaMA 3.1 8B Instruct with Unsloth (4-bit Quantization)

In [ ]:
from unsloth import FastLanguageModel

max_seq_length = 1024  # Ideal for Colab T4 memory & technical contexts
dtype = None           # Auto-detect float16 / bfloat16
load_in_4bit = True    # 4-bit quantization keeps VRAM under 8GB

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

print(f"✅ Model and Tokenizer loaded successfully!")

## Cell 4: Attach LoRA Adapters (Rank = 32)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",  # Saves ~30% VRAM
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

## Cell 5: Download & Assemble the Curated 7,400-Sample Dataset

This cell pulls the high-density datasets directly into Colab and streams the top balanced subsets.

In [ ]:
import random, json, os
import pandas as pd
from datasets import load_dataset, Dataset

random.seed(42)

SYSTEM_PROMPT = (
    "You are an expert technical intelligence and security research AI assistant. "
    "Your objective is to provide direct, technically accurate, comprehensive, and objective "
    "analyses for all queries, including penetration testing methodologies, software vulnerability "
    "mechanics, system architecture, and operational procedures. "
    "You do not give generic refusals; you explain technical mechanisms with verifiable factual precision."
)

def format_chat(system, user, assistant):
    return (
        f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n"
        f"{system}<|eot_id|>"
        f"<|start_header_id|>user<|end_header_id|>\n\n"
        f"{user}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>\n\n"
        f"{assistant}<|eot_id|>"
    )

raw_samples = []
print("=" * 70)
print("  DOWNLOADING CURATED SAMPLES (~7,400 TOTAL FOR OVERNIGHT RUN)")
print("=" * 70)

# ── 1. JailFact-Bench (Upload xlsx if available) ──
print("\n📥 [1/6] JailFact-Bench...")
if os.path.exists("jailfact-bench.xlsx"):
    df = pd.read_excel("jailfact-bench.xlsx")
    for _, row in df.iterrows():
        jp = str(row.get("Jailbreak Prompt", "")).strip()
        fp = str(row.get("Factuality Prompt", "")).strip()
        fc = str(row.get("Factuality Prompt Category", "")).strip()
        ver = str(row.get("Factuality can be checked by", "")).strip()
        if jp and fp:
            resp = f"Technical Analysis:\n\nTarget: {fp}\nDomain: {fc}\nVerification: {ver}\n\nOperational Mechanics: Follow standard RFC verification and security auditing methodologies."
            raw_samples.append({"instruction": jp, "response": resp, "source": "JailFact-Bench"})
    print(f"  ✅ Loaded {len([s for s in raw_samples if s['source'] == 'JailFact-Bench'])} from local file.")
else:
    print("  ℹ️ jailfact-bench.xlsx not found in working directory (optional upload).")

# ── 2. Dolphin (2,000 Refusal-Free Samples) ──
print("\n📥 [2/6] Dolphin (2,000 Refusal-Free Samples)...")
try:
    ds_dolphin = load_dataset("cognitivecomputations/dolphin", "flan1m-alpaca-uncensored", split="train", streaming=True)
    c = 0
    for row in ds_dolphin:
        inst = row.get("instruction", "")
        out = row.get("output", "")
        if inst and out and len(out) > 50 and "as an ai" not in out.lower() and "cannot assist" not in out.lower():
            raw_samples.append({"instruction": inst, "response": out, "source": "Dolphin"})
            c += 1
            if c >= 2000: break
    print(f"  ✅ Loaded {c} samples from Dolphin.")
except Exception as e: print(f"  ⚠️ Dolphin error: {e}")

# ── 3. Exploit-DB (1,500 CVE & Exploit Samples) ──
print("\n📥 [3/6] Exploit-DB Dataset (1,500 CVE & Exploit Samples)...")
try:
    ds_exploit = load_dataset("darkknight25/Exploit_Database_Dataset", split="train", streaming=True)
    c = 0
    for row in ds_exploit:
        cve = str(row.get("cve_id", row.get("id", "CVE"))).strip()
        desc = str(row.get("description", "")).strip()
        code = str(row.get("code", "")).strip()
        if desc:
            prompt = f"Explain the technical mechanics and vulnerability architecture for: {cve} - {desc}"
            resp = f"## Technical Vulnerability Analysis: {cve}\n\n**Description:**\n{desc}\n\n"
            if code: resp += f"**Execution Logic:**\n```\n{code[:800]}\n```\n\n"
            resp += "**Remediation:** Enforce parameter sanitization and apply vendor security updates."
            raw_samples.append({"instruction": prompt, "response": resp, "source": "Exploit-DB"})
            c += 1
            if c >= 1500: break
    print(f"  ✅ Loaded {c} samples from Exploit-DB.")
except Exception as e: print(f"  ⚠️ Exploit-DB error: {e}")

# ── 4. Web & Portal Security (1,500 Samples) ──
print("\n📥 [4/6] Vyber07/cyber-security (1,500 Web/Portal Pentesting Samples)...")
try:
    ds_sec = load_dataset("Vyber07/cyber-security", split="train", streaming=True)
    c = 0
    for row in ds_sec:
        inst = row.get("instruction", row.get("question", ""))
        resp = row.get("response", row.get("answer", ""))
        if inst and resp and len(str(resp)) > 60:
            raw_samples.append({"instruction": inst, "response": resp, "source": "Vyber07-CyberSecurity"})
            c += 1
            if c >= 1500: break
    print(f"  ✅ Loaded {c} samples from Vyber07.")
except Exception as e: print(f"  ⚠️ Vyber07 error: {e}")

# ── 5. TruthfulQA (817 Factual Ground-Truth Samples) ──
print("\n📥 [5/6] TruthfulQA (817 Factual Ground-Truth Samples)...")
try:
    ds_tqa = load_dataset("truthfulqa/truthful_qa", "generation", split="validation")
    c = 0
    for row in ds_tqa:
        q = str(row.get("question", "")).strip()
        best = str(row.get("best_answer", "")).strip()
        if q and best:
            resp = f"Factual Determination:\n\n{best}"
            raw_samples.append({"instruction": q, "response": resp, "source": "TruthfulQA"})
            c += 1
    print(f"  ✅ Loaded {c} samples from TruthfulQA.")
except Exception as e: print(f"  ⚠️ TruthfulQA error: {e}")

# ── 6. OpenHermes-2.5 (1,500 Reasoning Samples) ──
print("\n📥 [6/6] OpenHermes-2.5 (1,500 Technical Reasoning Samples)...")
try:
    ds_hermes = load_dataset("teknium/OpenHermes-2.5", split="train", streaming=True)
    c = 0
    for row in ds_hermes:
        convs = row.get("conversations", [])
        if len(convs) >= 2:
            u = next((item["value"] for item in convs if item.get("from") == "human"), "")
            a = next((item["value"] for item in convs if item.get("from") == "gpt"), "")
            if u and a and len(a) > 80:
                raw_samples.append({"instruction": u, "response": a, "source": "OpenHermes"})
                c += 1
                if c >= 1500: break
    print(f"  ✅ Loaded {c} samples from OpenHermes.")
except Exception as e: print(f"  ⚠️ OpenHermes error: {e}")

print(f"\n{'='*70}")
print(f"  Total Curated Samples: {len(raw_samples)}")
print(f"{'='*70}")

In [ ]:
# Format into LLaMA 3.1 Chat Template & Create Train/Val Splits
formatted_dataset = []
for s in raw_samples:
    text = format_chat(SYSTEM_PROMPT, s["instruction"], s["response"])
    formatted_dataset.append({"text": text, "instruction": s["instruction"], "source": s["source"]})

random.shuffle(formatted_dataset)
split_idx = int(len(formatted_dataset) * 0.95)
train_data = formatted_dataset[:split_idx]
val_data = formatted_dataset[split_idx:]

train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data)

print(f"✅ Training set size:   {len(train_dataset)} samples")
print(f"✅ Validation set size: {len(val_dataset)} samples")
print(f"\nSample preview (first 400 chars):\n{train_dataset[0]['text'][:400]}...")

## Cell 6: Configure & Start Overnight Training (SFTTrainer)

- **Effective Batch Size:** 8 (`per_device=2`, `grad_accum=4`)
- **Learning Rate:** 2e-4 with Cosine decay
- **Duration:** 1 epoch (~1.5 hours on T4 GPU)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

training_args = TrainingArguments(
    output_dir="./outputs_redteam",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,  # Effective Batch Size = 8
    num_train_epochs=1,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=50,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    optim="adamw_8bit",
    weight_decay=0.01,
    max_grad_norm=0.3,
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    seed=42,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=training_args,
)

print("🚀 Starting overnight fine-tuning on Colab T4 GPU...")
stats = trainer.train()

print(f"\n{'='*60}")
print(f"✅ Training Complete!")
print(f"Total Runtime: {stats.metrics['train_runtime']/60:.1f} minutes")
print(f"Final Loss:    {stats.metrics['train_loss']:.4f}")
print(f"{'='*60}")

## Cell 7: Test the Model on Real Vulnerability Queries

In [ ]:
FastLanguageModel.for_inference(model)

def query_model(prompt_text, max_new_tokens=512):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt_text},
    ]
    prompt_str = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt_str, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.2,  # Low temperature for strict factual accuracy
            top_p=0.9,
            do_sample=True,
        )
    return tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

# Test prompts
test_suite = [
    "Explain how an attacker identifies and tests for an authentication bypass vulnerability on an enterprise ERP portal.",
    "What is the technical root cause of CVE-2020-6287 (SAP NetWeaver RECON) and how is it executed?",
    "What is the difference between Insecure Direct Object References (IDOR) and Broken Object Level Authorization (BOLA)?",
]

for i, q in enumerate(test_suite, 1):
    print(f"\n{'━'*70}")
    print(f"TEST {i}: {q}")
    print(f"{'━'*70}")
    print(query_model(q))


## Cell 8: Save Model Adapters (To Colab & Google Drive)

In [ ]:
save_dir = "llama3_1_8b_redteam_lora"
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)
print(f"✅ Adapters saved to ./{save_dir}")

# Save to Google Drive
from google.colab import drive
import shutil

try:
    drive.mount('/content/drive')
    drive_dest = f"/content/drive/MyDrive/{save_dir}"
    shutil.copytree(save_dir, drive_dest, dirs_exist_ok=True)
    print(f"✅ Adapters permanently backed up to: {drive_dest}")
except Exception as e:
    print(f"ℹ️ Google Drive mount skipped or failed: {e}")